# Lección 12.1 — Atención: "¿a qué palabras debo prestar atención?"

> **Módulo 12 · Atención y Transformers** · Nivel 4 — Avanzado
> ⏱️ Tiempo estimado: 90–110 min (lección con ejercicios intercalados ≈ 65 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 11.2 (*embeddings*), Lección 11.5 (seq2seq y el cuello de botella)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar con una analogía qué problema resuelve la **atención** y por qué cambió el procesamiento de lenguaje.
- Calcular **a mano** (con numpy) una atención: puntajes → porcentajes (*softmax*) → promedio ponderado.
- Leer un **mapa de calor de atención** y explicar qué significa cada casilla.
- Predecir qué pasa cuando los puntajes son muy grandes o muy pequeños (atención "afilada" o "difusa").

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas y una función para dibujar **mapas de calor** (tablas de colores: más oscuro = número más grande).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=2, suppress=True)
plt.rcParams["figure.figsize"] = (6, 4)

def mapa_calor(M, filas, columnas, titulo="", ax=None):
    """Dibuja la matriz M como tabla de colores, con el número escrito en cada casilla."""
    ax = ax or plt.gca()
    ax.imshow(M, cmap="Blues")
    ax.set_xticks(range(len(columnas)), columnas, rotation=45, ha="right")
    ax.set_yticks(range(len(filas)), filas)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            claro = M[i, j] < M.min() + 0.6 * (M.max() - M.min())
            ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", fontsize=8,
                    color="black" if claro else "white")
    ax.set_title(titulo)

print("✅ Todo listo")

---
## 1. 📖 La intuición: tus ojos saltan a lo importante

Lee esta frase:

> *"El **gato** no cruzó la calle porque **estaba muy cansado**."*

¿Quién estaba cansado? El gato, claro. Para entender "cansado" tu cerebro **volvió atrás** y buscó en la frase a quién se refería. No le dio la misma importancia a todas las palabras: "gato" pesó mucho; "la" o "calle", casi nada.

Eso es **atención**: para entender una palabra, decidir **cuánto mirar a cada una de las demás**.

### ¿Por qué hacía falta? El cuello de botella
En la Lección 11.5 viste el seq2seq con RNN: el codificador lee toda la frase y la resume en **un solo vector** (la "maleta"). El decodificador solo tiene esa maleta para trabajar.

```
Sin atención:  [frase de 40 palabras] ──► 🎒 un solo vector ──► traducción
                                          (todo tiene que caber aquí)

Con atención:  [frase de 40 palabras] ◄──┐
                 ▲   ▲   ▲   ▲            │ en cada paso, el decodificador
                 └───┴───┴───┴── mira ────┘ vuelve a mirar TODAS las palabras
                                            y elige en cuáles fijarse
```

Es como hacer un examen **a libro abierto**: en lugar de memorizarlo todo en una mochila, puedes volver a consultar el libro en cada pregunta.

### ✋ Ejercicio 1 — Sé el mecanismo de atención
Para cada frase, ¿a qué palabra "mira" tu cerebro para entender la palabra en **negrita**?

1. *"El trofeo no cupo en la maleta porque **era** demasiado grande."*
2. *"El trofeo no cupo en la maleta porque **era** demasiado pequeña."*
3. *"María le prestó su libro a Ana porque **ella** ya lo había leído."*

✍️ *Tus respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Trofeo**: lo que es demasiado grande es lo que no cabe.
2. **Maleta**: cambiar *una sola palabra* al final cambia a qué hay que mirar. ¡Y en español hay una pista extra: "pequeñ**a**" es femenino, como "maleta"!
3. **María**: quien presta el libro es quien ya lo leyó.

La moraleja: **a qué mirar depende de toda la frase**, incluso de palabras que vienen *después*. Por eso un modelo necesita poder mirar a todas las posiciones, no solo a la anterior.
</details>

---
## 2. 📖 El concepto: atención = un promedio ponderado

La atención hace siempre **tres pasos**:

| Paso | Qué hace | En la frase del gato |
|---|---|---|
| 1. **Puntuar** | Da un puntaje de relevancia a cada palabra | gato: alto · calle: bajo |
| 2. **Convertir a porcentajes** | Transforma los puntajes en pesos positivos que suman 100 % (*softmax*) | gato 81 % · calle 4 % … |
| 3. **Mezclar** | Hace un **promedio ponderado** de la información de las palabras con esos pesos | "cansado" se queda con 81 % de la información de "gato" |

El paso 2 lo hace una función llamada **softmax**. No necesitas la fórmula: convierte cualquier lista de números en porcentajes, y los números más altos se llevan una parte **desproporcionadamente** mayor.

In [ ]:
def softmax(puntajes):
    """Convierte puntajes en pesos positivos que suman 1."""
    e = np.exp(puntajes - np.max(puntajes))   # restar el máximo no cambia el resultado; evita números gigantes
    return e / e.sum()

puntajes = np.array([2.0, 1.0, 0.0])
pesos = softmax(puntajes)
print("Puntajes:", puntajes)
print("Pesos   :", pesos, "  → suman", round(pesos.sum(), 3))

### 🔮 Ejercicio 2 — Juega con softmax
Antes de ejecutar la celda de abajo, predice:
1. Si a **todos** los puntajes les **sumo 10** (`[12, 11, 10]`), ¿cambian los pesos?
2. Si **multiplico** todos los puntajes **por 10** (`[20, 10, 0]`), ¿los pesos se vuelven más parejos o más extremos?
3. Si todos los puntajes son **iguales** (`[5, 5, 5]`), ¿qué pesos salen?

✍️ *Mis predicciones:*

In [ ]:
for p in ([12, 11, 10], [20, 10, 0], [5, 5, 5]):     # ✏️ prueba tus propias listas
    print(f"puntajes {str(p):<14} → pesos {softmax(np.array(p, dtype=float))}")

<details><summary><b>👀 Ver respuesta</b></summary>

1. **No cambian** (0.67, 0.24, 0.09, igual que antes). Softmax solo se fija en las **diferencias** entre puntajes, no en su valor absoluto.
2. **Más extremos**: 1.00, 0.00, 0.00. Al multiplicar, las diferencias crecen y el ganador se lo lleva casi todo.
3. **Iguales**: 0.33 cada uno. Si nada destaca, la atención se reparte por igual.

Guarda la idea 2: **puntajes grandes = atención afilada**. Volverá en la Sección 4.
</details>

---
## 3. 📖 Cómo funciona por dentro: puntuar con el producto punto

¿De dónde salen los puntajes? Cada palabra es un **vector** (su *embedding*, Lección 11.2). Para medir qué tanto encaja una palabra con lo que buscamos, se usa el **producto punto**: multiplicar casilla por casilla y sumar.

- Si dos vectores apuntan a lo mismo → producto punto **alto**.
- Si no tienen nada en común → **cero** (o negativo).

Vamos a inventar vectores pequeños de 4 casillas, donde cada casilla significa algo fácil de leer: **[ser vivo, lugar, acción, estado]**. (Los modelos reales tienen cientos de casillas y nadie las elige a mano: se aprenden).

In [ ]:
palabras = ["gato", "cruzó", "calle", "estaba", "cansado"]
#                 ser vivo  lugar  acción  estado
X = np.array([[1.0,      0.0,   0.0,    0.2],    # gato
              [0.0,      0.0,   1.0,    0.0],    # cruzó
              [0.0,      1.0,   0.0,    0.0],    # calle
              [0.0,      0.0,   0.5,    0.5],    # estaba
              [0.2,      0.0,   0.0,    1.0]])   # cansado

# Lo que "cansado" BUSCA: un ser vivo (casilla 1). El 3 indica "con mucha fuerza".
consulta = np.array([3.0, 0.0, 0.0, 0.0])

puntajes = X @ consulta            # producto punto de la consulta con cada palabra
pesos = softmax(puntajes)
for w, s, p in zip(palabras, puntajes, pesos):
    print(f"{w:<8} puntaje = {s:4.1f}   peso = {p:5.1%}  " + "█" * int(p * 40))

### ✋ Ejercicio 3 — El producto punto a mano
Con la consulta `[3, 0, 0, 0]`:
1. Calcula a mano el puntaje de **"gato"** `[1, 0, 0, 0.2]` y de **"cansado"** `[0.2, 0, 0, 1]`.
2. ¿Por qué "cruzó", "calle" y "estaba" tienen exactamente el mismo peso?

✍️ *Tus respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. gato: 3·1 + 0·0 + 0·0 + 0·0.2 = **3.0**. cansado: 3·0.2 + 0 + 0 + 0 = **0.6**.
2. Porque las tres tienen un 0 en la casilla "ser vivo", la única que le importa a la consulta: puntaje 0 → mismo peso (≈4 % cada una). "gato" se lleva ≈81 %.
</details>

### 📖 Paso 3: mezclar
Ya tenemos los pesos. El resultado de la atención es el **promedio ponderado** de los vectores de las palabras: una nueva representación de "cansado" que ahora **contiene información del gato**.

In [ ]:
resultado = pesos @ X       # suma de (peso de cada palabra × su vector)
print("Vector original de 'cansado'  :", X[4])
print("Resultado de la atención      :", resultado)

### 🔮 Ejercicio 4 — ¿Qué casilla creció?
Compara los dos vectores de arriba. ¿Qué casilla del resultado es la más grande ahora? ¿Qué significa eso en palabras?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

La casilla **"ser vivo"** pasó de 0.2 a ≈0.82. El nuevo vector de "cansado" ahora "sabe" que se refiere a un ser vivo: absorbió información del gato. Eso es exactamente lo que hace la atención: **enriquecer cada palabra con el contexto que le importa**.
</details>

### 📖 Todas las palabras a la vez: el mapa de atención
Hasta ahora solo "cansado" preguntó. En un Transformer **todas** las palabras preguntan a la vez. Un primer intento sencillo: que cada palabra use **su propio vector** como consulta. El resultado es una tabla (filas = quien pregunta, columnas = a quién mira) que se dibuja como un **mapa de calor**.

### 🔮 Ejercicio 5 — ¿Dónde estará la casilla más oscura?
Antes de ejecutar: si cada palabra se compara **con su propio vector**, ¿qué patrón esperas ver en el mapa? ¿Mirará "cansado" principalmente a "gato"?

✍️ *Mi predicción:*

In [ ]:
puntajes_todos = X @ X.T                     # cada palabra contra cada palabra
pesos_todos = np.array([softmax(fila) for fila in puntajes_todos])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
mapa_calor(puntajes_todos, palabras, palabras, "Puntajes (producto punto)", axes[0])
mapa_calor(pesos_todos, palabras, palabras, "Pesos de atención (cada fila suma 1)", axes[1])
axes[0].set_ylabel("quien pregunta"); axes[1].set_xlabel("a quién mira")
plt.tight_layout(); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

La **diagonal** es la más oscura (en "estaba" empata con dos vecinas): cada palabra se parece sobre todo **a sí misma**. "cansado" se mira a sí mismo con un 36 % y a "gato" solo con un 19 %.

Esto es un problema: usar el mismo vector para "lo que busco" y "lo que soy" hace que cada palabra se mire al espejo. "cansado" **es** un estado, pero **busca** un ser vivo. En la Lección 12.2 resolveremos esto dándole a cada palabra **tres papeles distintos**: consulta, clave y valor.
</details>

---
## 4. 📖 ¿Qué pasa si…? Atención afilada vs. difusa

En el Ejercicio 2 viste que multiplicar los puntajes cambia qué tan "extremos" son los pesos. En la atención esto tiene nombre:

- **Puntajes grandes** → atención **afilada**: casi todo el peso a una sola palabra (como buscar con un láser).
- **Puntajes pequeños** → atención **difusa**: un promedio casi parejo de todo (como una linterna que ilumina toda la habitación).

Al factor que multiplica los puntajes lo llamaremos **nitidez**.

### 🔮 Ejercicio 6 — Predice las tres barras
Vamos a repetir la consulta de "cansado" con nitidez **0.1**, **1** y **5**. ¿Cuánto peso crees que recibirá "gato" en cada caso? (Con nitidez 1 ya sabes que es ≈81 %).

✍️ *Mis predicciones:* 0.1 → ___ · 1 → 81 % · 5 → ___

In [ ]:
NITIDECES = [0.1, 1, 5]      # ✏️ prueba otros valores, por ejemplo 0.01 o 20

fig, axes = plt.subplots(1, len(NITIDECES), figsize=(4 * len(NITIDECES), 3), sharey=True)
for ax, k in zip(axes, NITIDECES):
    p = softmax(k * (X @ consulta))
    ax.bar(palabras, p, color="tab:blue")
    ax.set_title(f"nitidez = {k}  →  gato {p[0]:.0%}")
    ax.tick_params(axis="x", rotation=45)
axes[0].set_ylabel("peso de atención")
plt.tight_layout(); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

- Nitidez 0.1 → gato ≈ **25 %**: casi un promedio parejo (el mínimo posible sería 20 % con 5 palabras).
- Nitidez 1 → gato ≈ **81 %**.
- Nitidez 5 → gato ≈ **100 %**: la atención se vuelve una búsqueda "dura" de una sola palabra.

¿Cuál es mejor? Ninguna siempre. Una atención demasiado afilada deja de aprender bien: si un peso es ~100 % y el resto ~0 %, los gradientes (Lección 9.4) casi desaparecen. Por eso los Transformers **dividen los puntajes por un número** para mantenerlos en un rango razonable (lo verás en la Lección 12.2).
</details>

### 📖 Atención en la traducción: el modelo aprende a reordenar
Así nació la atención (Bahdanau y colegas, 2014): para traducir. Cuando el decodificador escribe cada palabra en español, **mira** a las palabras en inglés.

Inventemos vectores donde las palabras con el mismo significado se parecen (en un modelo real esto se aprende con millones de frases):

In [ ]:
rng = np.random.default_rng(0)
significados = np.eye(4)                    # 4 significados: artículo, gato, negro, dormir
ingles  = ["the", "black", "cat", "sleeps"]
espanol = ["el", "gato", "negro", "duerme"]
# cada palabra = su significado + un poco de ruido
claves_en = significados[[0, 2, 1, 3]] + rng.normal(0, 0.15, (4, 4))
consultas_es = significados[[0, 1, 2, 3]] + rng.normal(0, 0.15, (4, 4))

alineacion = np.array([softmax(3 * (consultas_es @ claves_en.T)[i]) for i in range(4)])

### 🔮 Ejercicio 7 — Dibuja el mapa antes de verlo
En inglés el adjetivo va **antes** ("black cat") y en español **después** ("gato negro"). Cuando el modelo escribe "gato", ¿a qué palabra inglesa mirará? Dibuja en papel la tabla 4×4 marcando las casillas oscuras.

✍️ *Mi dibujo:*

In [ ]:
mapa_calor(alineacion, espanol, ingles, "Al escribir cada palabra en español, ¿a qué palabra inglesa mira?")
plt.ylabel("escribiendo…"); plt.xlabel("mirando a…"); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

Casi una diagonal, pero con una **cruz** en el medio: "gato" mira a "cat" (3.ª columna) y "negro" mira a "black" (2.ª columna). Cada casilla oscura se lleva entre 0.84 y 0.88 del peso.

Un seq2seq sin atención tendría que "recordar" ese cambio de orden dentro de la maleta. Con atención, el modelo simplemente **mira en el lugar correcto** en cada paso, sin importar el orden.
</details>

---
## 5. 📖 Las perillas de la atención

La atención básica tiene pocas perillas; las importantes llegarán con los Transformers (Lección 12.3).

| Perilla | Qué controla | Si la **subes** ⬆️ | Si la **bajas** ⬇️ | Valor típico |
|---|---|---|---|---|
| Nitidez (escala de los puntajes) | Qué tan concentrada es la atención | Atención afilada, casi "dura"; gradientes débiles | Atención difusa, casi un promedio | 1/√d (Lección 12.2) |
| Dimensión de los vectores (*d*) | Cuánta información cabe en cada palabra | Más matices, más cálculo | Más barato, menos expresivo | 64–128 por cabeza |
| Longitud del texto (*n*) | Cuántas palabras se miran | Más contexto, costo × n² | Más barato, menos contexto | 1k–100k+ tokens |

### 🔮 Ejercicio 8 — El costo de mirarlo todo
Cada palabra mira a **todas** las demás. Si un texto tiene *n* palabras, ¿cuántas casillas tiene el mapa de atención? Predice: si el texto pasa de 1,000 a 10,000 palabras (×10), ¿cuántas veces más casillas hay?

✍️ *Mi predicción:*

In [ ]:
for n in [10, 1_000, 10_000, 100_000]:      # ✏️
    print(f"{n:>7,} palabras → {n * n:>18,} casillas en el mapa de atención")

<details><summary><b>👀 Ver respuesta</b></summary>

El mapa tiene **n × n** casillas: ×10 palabras = **×100** casillas. Con 100,000 palabras son 10,000 millones de casillas *por cada cabeza y cada capa*. Este crecimiento **cuadrático** es el gran costo de la atención y la razón por la que los modelos tienen una **ventana de contexto** máxima (Lección 13.9).
</details>

---
## 6. 📖 Errores comunes y malentendidos

| 🚨 Malentendido | La realidad |
|---|---|
| "La atención es una explicación de por qué el modelo decidió algo" | Es una **pista**, no una prueba. El modelo también transforma la información en otras partes (Lección 12.3). |
| "Si una palabra no importa, recibe peso 0" | Softmax **nunca** da exactamente 0, y los pesos **siempre suman 1**: el modelo tiene que poner la atención en algún lado, aunque nada sea relevante. |
| "La atención solo sirve para texto" | También se usa en imágenes, audio, proteínas y datos tabulares (Lección 10.6, Vision Transformers). |
| "Más contexto siempre es gratis" | El costo crece con n² (Ejercicio 8). |
| "La atención reemplaza a los *embeddings*" | La atención **mezcla** vectores que ya existen; sin buenos *embeddings* no tiene nada útil que mezclar. |

### 🔮 Ejercicio 9 — ¿Y si nada es relevante?
La consulta busca algo que **ninguna** palabra tiene (un vector de ceros). ¿Qué pesos esperas? ¿Puede la atención "no mirar a nadie"?

✍️ *Mi predicción:*

In [ ]:
consulta_vacia = np.zeros(4)
print(softmax(X @ consulta_vacia))

<details><summary><b>👀 Ver respuesta</b></summary>

Pesos de **0.2** para cada una de las 5 palabras: un promedio parejo. La atención **no puede** abstenerse; como los pesos deben sumar 1, reparte por igual. En la Lección 12.2 verás que GPT-2 aprendió un truco para esto: cuando no le interesa nada, "estaciona" su atención en el **primer token** de la frase.
</details>

---
## 7. 📋 Ficha del mecanismo: atención

| | Atención |
|---|---|
| **Qué es** | Un mecanismo que, para cada elemento, hace un promedio ponderado de los demás elementos según su relevancia |
| **Intuición** | Examen a libro abierto: en cada momento vuelves a mirar las partes del texto que importan |
| **Cómo funciona** | Puntuar (producto punto) → porcentajes (softmax) → promedio ponderado |
| **Qué se aprende** | Los vectores (y, en la Lección 12.2, las matrices que crean consultas, claves y valores) |
| **✅ Fortalezas** | Mira cualquier posición sin importar la distancia · Elimina el cuello de botella del seq2seq · Se calcula en paralelo · Se puede visualizar |
| **❌ Debilidades** | Costo n² con la longitud · No sabe el orden de las palabras por sí sola (Lección 12.3) · Sus pesos no son una explicación completa |
| **Dónde la verás** | Traductores, Transformers, GPT, BERT, Vision Transformers, modelos de difusión |

## 8. 🧠 Resumen en 5 puntos
1. La **atención** decide, para cada palabra, **cuánto mirar** a cada una de las demás.
2. Funciona en tres pasos: **puntuar** (producto punto) → **porcentajes** (softmax) → **promedio ponderado**.
3. El resultado es un vector **enriquecido con contexto**: "cansado" absorbe información de "gato".
4. Puntajes grandes dan atención **afilada**; pequeños, **difusa**. Los pesos siempre suman 1.
5. Resolvió el **cuello de botella** del seq2seq, pero cuesta **n²** con la longitud del texto.

---
## 9. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas** (≈30 min).

### 9.1 Repaso rápido (opción múltiple)

**1.** ¿Qué problema del seq2seq con RNN (Lección 11.5) resolvió la atención?
- a) Que entrenaba muy rápido · b) Que tenía que resumir toda la frase en un solo vector · c) Que no usaba *embeddings* · d) Que no podía traducir

**2.** Los puntajes de atención son `[4, 4, 4, 4]`. ¿Cuáles son los pesos?
- a) `[1, 0, 0, 0]` · b) `[0.25, 0.25, 0.25, 0.25]` · c) `[4, 4, 4, 4]` · d) `[0, 0, 0, 0]`

**3.** Multiplicas todos los puntajes por 100. La atención se vuelve…
- a) más difusa · b) más afilada · c) igual · d) negativa

**4.** Un texto pasa de 2,000 a 8,000 tokens. ¿Cuántas veces crece el mapa de atención?
- a) ×4 · b) ×8 · c) ×16 · d) ×64

**5.** ¿Cuál afirmación es **falsa**?
- a) Los pesos de atención de una fila suman 1 · b) La atención puede mirar palabras lejanas tan fácilmente como las cercanas · c) Un peso alto prueba que esa palabra causó la predicción · d) El producto punto mide qué tanto se parecen dos vectores

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** El cuello de botella: todo tenía que caber en un vector.
2. **b)** Puntajes iguales → pesos iguales.
3. **b)** Diferencias más grandes → softmax más extremo.
4. **c)** 4 veces más tokens → 4 × 4 = 16 veces más casillas.
5. **c)** La atención es una pista, no una prueba de causalidad.
</details>

### 9.2 Preguntas para pensar (sin ejecutar)

**A.** Un RNN lee la frase palabra por palabra. Para relacionar la palabra 1 con la palabra 50, la información tiene que pasar por 49 pasos intermedios. ¿Cuántos pasos necesita la atención? ¿Por qué eso ayuda con las frases largas?

**B. Caso 💼** Un hospital usa un modelo con atención para leer notas médicas y marcar riesgo de infarto. Un médico ve en el mapa de calor que el modelo "miró" mucho la palabra *"fumador"* y concluye: *"el modelo decidió por el tabaco"*. ¿Qué le responderías?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** **Un solo paso**: la palabra 50 mira directamente a la palabra 1. En un RNN la información se va "diluyendo" en cada paso (la memoria corta de la Lección 11.3); con atención, la distancia no importa.

**B.** Que es una **pista razonable, pero no una prueba**. La atención muestra de dónde se mezcló información en *una* capa, pero el modelo tiene muchas capas y otras transformaciones. Para afirmar que "decidió por el tabaco" habría que hacer un experimento: quitar o cambiar la palabra y ver si la predicción cambia (es la idea de la importancia por permutación de la Lección 7.6).
</details>

### 9.3 🧩 Práctica integradora: la cafetería que "atiende"

Una cafetería quiere un asistente que, dada la descripción de lo que un cliente quiere, **estime el precio** de lo que va a pedir. En lugar de elegir un solo producto, usará **atención**: mezclará los precios de los productos según qué tanto encajan con el pedido.

Cada producto tiene un vector **[dulce, fuerte, frío]**. Lo harás en 5 pasos cortos; **no avances sin revisar cada respuesta**.

**Paso 1 ▶️ — El menú**

In [ ]:
productos = ["espresso", "latte", "frappé", "americano", "té helado"]
#                       dulce  fuerte  frío
menu = np.array([[0.0,   1.0,   0.0],
                 [0.6,   0.4,   0.0],
                 [1.0,   0.2,   1.0],
                 [0.0,   0.7,   0.0],
                 [0.5,   0.0,   1.0]])
precios = np.array([30, 45, 60, 35, 40])

# Pedido del cliente: "quiero algo FRÍO y un poco dulce"
pedido = np.array([1.0, 0.0, 2.0])
print("✅ Menú cargado")

**Paso 2 🔮 — Puntajes.** Antes de ejecutar, calcula a mano el puntaje (producto punto) del **frappé** y del **espresso** con el pedido `[1, 0, 2]`. ¿Qué producto tendrá el puntaje más alto?

✍️ *Mi cálculo:*

In [ ]:
puntajes_cafe = menu @ pedido
for prod, s in zip(productos, puntajes_cafe):
    print(f"{prod:<10} puntaje = {s:.1f}")

<details><summary><b>👀 Ver respuesta</b></summary>

- Frappé: 1·1 + 0·0.2 + 2·1 = **3.0** (el más alto).
- Espresso: 1·0 + 0·1 + 2·0 = **0.0**.
- El té helado queda segundo (2.5) y el latte tercero (0.6).
</details>

**Paso 3 ▶️ — Pesos y precio estimado.** Convierte los puntajes en pesos y calcula el precio como promedio ponderado.

In [ ]:
pesos_cafe = softmax(puntajes_cafe)
for prod, p in zip(productos, pesos_cafe):
    print(f"{prod:<10} peso = {p:5.1%}")
print(f"\nPrecio estimado = ${pesos_cafe @ precios:.2f}")

✋ ¿Por qué el precio estimado **no** es exactamente el del frappé ($60)?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Porque la atención **no elige** un producto: **mezcla**. El frappé se lleva ≈56 % del peso y el té helado ≈34 %, así que el precio queda entre los dos: **≈ $50.97**. Es una búsqueda "suave".
</details>

**Paso 4 ✏️ — Gira la perilla de nitidez.** Cambia `NITIDEZ` a **10** y luego a **0.01**. Anota el precio estimado en cada caso.

In [ ]:
NITIDEZ = 10      # ✏️ prueba 10 y luego 0.01

p = softmax(NITIDEZ * puntajes_cafe)
print(f"Nitidez {NITIDEZ}: pesos = {p}  →  precio estimado = ${p @ precios:.2f}")

✍️ *Mis resultados:* nitidez 10 → $___ · nitidez 0.01 → $___

<details><summary><b>👀 Ver respuesta</b></summary>

- **Nitidez 10** → ≈ **$59.87**: casi todo el peso va al frappé. Es una búsqueda "dura" (como un diccionario: buscas una clave y sacas un solo valor).
- **Nitidez 0.01** → ≈ **$42.10**: casi el promedio simple del menú ($42). La atención ya no distingue entre productos.

La atención es un punto intermedio **ajustable** entre "elegir uno" y "promediar todo".
</details>

**Paso 5 ✋ — Conéctalo con la Lección 12.2.** En esta cafetería usamos tres cosas distintas:
- el **pedido** del cliente,
- los **vectores del menú** (con los que se compara el pedido),
- los **precios** (lo que se mezcla al final).

¿Por qué crees que es útil que los vectores con los que comparamos y lo que mezclamos (los precios) sean cosas **diferentes**?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Porque **lo que sirve para encontrar** algo no es lo mismo que **lo que quieres sacar** de él. Buscas un café por su sabor, pero lo que te llevas es su precio. En la Lección 12.2 esto tendrá nombre: el pedido es la **consulta** (*query*), los vectores del menú son las **claves** (*keys*) y los precios son los **valores** (*values*). Es justo lo que nos faltaba en el Ejercicio 5, donde cada palabra usaba el mismo vector para todo y terminaba mirándose al espejo.
</details>

---
## 📚 Para profundizar
- 📄 **Artículo original:** Bahdanau, Cho y Bengio (2014), *Neural Machine Translation by Jointly Learning to Align and Translate*: la primera atención, para traducir.
- 📺 **Video:** 3Blue1Brown, *Attention in transformers, step-by-step* (inglés, con subtítulos): animaciones excelentes.
- 📘 **Blog:** Jay Alammar, *Visualizing A Neural Machine Translation Model* (inglés).

---
⬅️ **Anterior:** [11.6 Proyecto del módulo: ¿Cuántas bicis mañana? 🚲](../m11-secuencias/11.6-proyecto-del-modulo.md) · ➡️ **Siguiente:** [12.2 Auto-atención: consultas, claves y valores (la biblioteca)](12.2-auto-atencion-qkv.ipynb)